# UrbanRide — Synthetic Data Generator
## Databricks Lakehouse · Medallion Architecture

**What this notebook does:**
1. Creates Unity Catalog: `urbanride` with schemas `bronze`, `silver`, `gold`
2. Installs dependencies
3. Generates 6 months of realistic ride-hailing data
4. Writes all raw files to DBFS (simulating Bronze ingestion landing zone)
5. Registers Bronze Delta tables pointing to those files

**Data period:** Last 6 months from today  
**Cities:** Delhi · Mumbai · Bengaluru · Hyderabad · Pune  
**Scale:** ~120K trips/day · ~250K events/day · ~300K total customers

> Run cells top to bottom. Cell 2 installs Faker — kernel restarts automatically.

In [0]:
# Faker is not pre-installed on Databricks serverless
# Install it first — this may restart the Python kernel, just re-run from Cell 2
%pip install faker --quiet
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import random
import json
import os
import gc
import uuid
import math
from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta

import pandas as pd
import numpy as np
from faker import Faker

# ── REPRODUCIBILITY ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker('en_IN')
Faker.seed(SEED)

# ── DATE RANGE — last 6 months ────────────────────────────────────────────
END_DATE   = date(2026,3,1)
START_DATE = date(2025,9,1)
TOTAL_DAYS = (END_DATE - START_DATE).days + 1

# ── SCALE ────────────────────────────────────────────────────────────────
TRIPS_PER_DAY_BASE      = 120_000
EVENTS_PER_DAY_BASE     = 250_000
TOTAL_CUSTOMERS         = 300_000
TOTAL_DRIVERS           = 20_000
NEW_CUSTOMERS_PER_DAY   = 500
CHUNK_SIZE              = 5_000    # rows per write chunk — prevents OOM

# ── CATALOG / SCHEMA ─────────────────────────────────────────────────────
CATALOG    = 'urbanride'
BRONZE     = f'{CATALOG}.bronze'
SILVER     = f'{CATALOG}.silver'
GOLD       = f'{CATALOG}.gold'

# Correct — Unity Catalog Volume
BASE_PATH = '/Volumes/urbanride/bronze/raw_files'

# ── CITY CONFIG ──────────────────────────────────────────────────────────
CITIES = ['Delhi','Mumbai','Bengaluru','Hyderabad','Pune']
CITY_WEIGHTS = [0.30, 0.25, 0.20, 0.15, 0.10]
CITY_CHURN   = {'Delhi':0.15,'Mumbai':0.18,'Bengaluru':0.20,'Hyderabad':0.22,'Pune':0.30}

# ── DIRTY CITY NAMES for data quality issues ──────────────────────────────
DIRTY_CITY = {
    'Bengaluru': ['Bangalore','BLR','bengaluru','Bengaluru'],
    'Hyderabad': ['Hyd','hyderabad','Hyderbad','Hyderabad'],
    'Delhi':     ['delhi','New Delhi','Delhi'],
    'Mumbai':    ['mumbai','Bombay','Mumbai'],
    'Pune':      ['pune','Pune'],
}

# ── INDIAN PUBLIC HOLIDAYS (approximate) ─────────────────────────────────
# These dates see higher demand (1.3x multiplier)
# def get_indian_holidays(year):
#     return {
#         date(year, 1, 1),   # New Year
#         date(year, 1, 14),  # Makar Sankranti
#         date(year, 1, 26),  # Republic Day
#         date(year, 3, 25),  # Holi (approx)
#         date(year, 4, 14),  # Dr. Ambedkar Jayanti
#         date(year, 8, 15),  # Independence Day
#         date(year, 10, 2),  # Gandhi Jayanti
#         date(year, 10, 20), # Dussehra (approx)
#         date(year, 11, 1),  # Diwali (approx)
#         date(year, 11, 2),  # Diwali day 2
#         date(year, 12, 25), # Christmas
#     }

def get_indian_holidays():
    """
    Accurate Indian public holidays for Sep 2025 — Mar 2026.
    Includes national holidays + major festivals that drive ride demand.
    New Year's Eve included for late-night surge effect.
    """
    return {
        # Sep 2025
        date(2025, 9, 5),   # Teacher's Day — high urban usage
        date(2025, 9, 7),   # Ganesh Chaturthi — massive in Mumbai, Pune, Bengaluru

        # Oct 2025
        date(2025, 10, 2),  # Gandhi Jayanti + Dussehra (coincide in 2025)
        date(2025, 10, 20), # Diwali — Lakshmi Puja (peak demand day)
        date(2025, 10, 21), # Diwali Day 2
        date(2025, 10, 22), # Diwali Day 3 — Govardhan Puja

        # Nov 2025
        date(2025, 11, 5),  # Guru Nanak Jayanti

        # Dec 2025
        date(2025, 12, 25), # Christmas
        date(2025, 12, 31), # New Year's Eve — late-night surge

        # Jan 2026
        date(2026, 1, 1),   # New Year's Day
        date(2026, 1, 14),  # Makar Sankranti — Pongal in Chennai/South India
        date(2026, 1, 26),  # Republic Day — national holiday

        # Feb 2026
        date(2026, 2, 26),  # Maha Shivaratri

        # Mar 2026 (just before END_DATE)
        date(2026, 3, 1),   # Holi (falls on Mar 1 in 2026)
    }

# HOLIDAYS = get_indian_holidays(START_DATE.year) | get_indian_holidays(END_DATE.year)
HOLIDAYS = get_indian_holidays()

def is_weekend(d): return d.weekday() >= 5      # Sat=5, Sun=6
def is_holiday(d): return d in HOLIDAYS

# def day_demand_multiplier(d):
#     """Adjust daily trip/event volume based on day type."""
#     if is_holiday(d):  return 1.30   # holidays = 30% more rides
#     if d.weekday() == 4: return 1.15  # Friday evening surge
#     if is_weekend(d):  return 0.75   # weekends = fewer commuter trips
#     return 1.0                        # regular weekday = baseline
def day_demand_multiplier(d):
    if d == date(2025, 12, 31): return 1.50   # NYE — highest demand night
    if d == date(2025, 10, 20): return 1.45   # Diwali peak
    if d in {date(2025,10,21), date(2025,10,22)}: return 1.35  # Diwali days
    if d == date(2025, 9, 7):   return 1.35   # Ganesh Chaturthi
    if is_holiday(d):           return 1.30   # all other holidays
    if d.weekday() == 4:        return 1.15   # Friday
    if is_weekend(d):           return 0.75   # Sat/Sun
    return 1.0

print(f'Date range  : {START_DATE}  to  {END_DATE}  ({TOTAL_DAYS} days)')
print(f'Holidays    : {len(HOLIDAYS)} Indian public holidays in range')
print(f'Weekends    : {sum(1 for i in range(TOTAL_DAYS) if is_weekend(START_DATE+timedelta(i)))} weekend days')
print(f'Catalog     : {CATALOG}')
print(f'Base path   : {BASE_PATH}')
print('Config ready.')


Date range  : 2025-09-01  to  2026-03-01  (182 days)
Holidays    : 14 Indian public holidays in range
Weekends    : 52 weekend days
Catalog     : urbanride
Base path   : /Volumes/urbanride/bronze/raw_files
Config ready.


In [0]:
# ── CREATE UNITY CATALOG ─────────────────────────────────────────────────
print('Creating Unity Catalog and schemas...')

spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
spark.sql(f'USE CATALOG {CATALOG}')

for schema in ['bronze','silver','gold','landing']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}')
    print(f'  Schema created: {CATALOG}.{schema}')

# ── CREATE VOLUME ─────────────────────────────────────────────────────────
# Volume must be created BEFORE any dbutils.fs operations inside it
spark.sql(f'CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.raw_files')
print(f'  Volume created : {CATALOG}.bronze.raw_files')

# ── CREATE DIRECTORIES INSIDE THE VOLUME ───────────────────────────────────────────────
dirs = [
    f'{BASE_PATH}/customers',
    f'{BASE_PATH}/drivers',
    f'{BASE_PATH}/trips',
    f'{BASE_PATH}/payments',
    f'{BASE_PATH}/clickstream',
]
for d in dirs:
    dbutils.fs.mkdirs(d)
    print(f'  Directory created: {d}')

print()
print('Catalog structure ready:')
print(f'  {CATALOG}.bronze  — raw Delta tables (registered after generation)')
print(f'  {CATALOG}.silver  — cleaned + validated (built in next notebooks)')
print(f'  {CATALOG}.gold    — aggregated + ML features (built in next notebooks)')


Creating Unity Catalog and schemas...
  Schema created: urbanride.bronze
  Schema created: urbanride.silver
  Schema created: urbanride.gold
  Schema created: urbanride.landing
  Volume created : urbanride.bronze.raw_files
  Directory created: /Volumes/urbanride/bronze/raw_files/customers
  Directory created: /Volumes/urbanride/bronze/raw_files/drivers
  Directory created: /Volumes/urbanride/bronze/raw_files/trips
  Directory created: /Volumes/urbanride/bronze/raw_files/payments
  Directory created: /Volumes/urbanride/bronze/raw_files/clickstream

Catalog structure ready:
  urbanride.bronze  — raw Delta tables (registered after generation)
  urbanride.silver  — cleaned + validated (built in next notebooks)
  urbanride.gold    — aggregated + ML features (built in next notebooks)


In [0]:
# ── SURGE MULTIPLIER (peak-hour aware) ───────────────────────────────────
def get_surge(hour, weather):
    """Surge correlated with time-of-day and weather."""
    if 8 <= hour <= 10:   dist = [0.40, 0.35, 0.25]   # morning peak
    elif 18 <= hour <= 21: dist = [0.35, 0.40, 0.25]  # evening peak
    elif 0 <= hour <= 6:   dist = [0.90, 0.08, 0.02]  # off-peak night
    else:                  dist = [0.75, 0.20, 0.05]  # mid-day
    surge = random.choices([1.0, 1.5, 2.0], weights=dist)[0]
    if weather == 'Rain': surge = min(surge + 0.2, 2.5)  # rain bumps surge
    return surge

# ── WEATHER ───────────────────────────────────────────────────────────────
def get_weather(city, month):
    """Seasonal weather — monsoon months heavier rain."""
    monsoon = month in [6,7,8,9]
    if monsoon:
        if city in ['Mumbai','Bengaluru','Hyderabad','Pune']:
            return random.choices(['Clear','Rain','Fog'],[0.35,0.55,0.10])[0]
        return random.choices(['Clear','Rain','Fog'],[0.50,0.40,0.10])[0]
    return random.choices(['Clear','Rain','Fog'],[0.70,0.20,0.10])[0]

# ── FARE CALCULATION ──────────────────────────────────────────────────────
def calc_fare(distance_km, surge, vehicle_type):
    base = {'Auto':25,'Sedan':30,'SUV':40}
    rate = {'Auto':9,'Sedan':12,'SUV':16}
    fare = base[vehicle_type] + rate[vehicle_type] * distance_km
    fare *= surge
    return round(max(30, fare + random.uniform(-10,10)), 2)

# ── DIRTY CITY NAME ───────────────────────────────────────────────────────
def dirty_city(city):
    """1-in-5 chance of returning a dirty city name variant."""
    if random.random() < 0.20:
        variants = DIRTY_CITY.get(city, [city])
        return random.choice(variants)
    return city

# ── CHUNK WRITE TO DBFS ───────────────────────────────────────────────────
def write_csv_to_dbfs(rows, volume_path):
    """Write list of dicts as CSV directly to UC Volume."""
    if not rows: return 0
    df = pd.DataFrame(rows)
    # Write directly to /Volumes/ path — no file: protocol needed
    df.to_csv(volume_path, index=False)
    return len(df)

def write_jsonl_to_dbfs(rows, volume_path):
    """Write list of dicts as newline-delimited JSON directly to UC Volume."""
    if not rows: return 0
    with open(volume_path, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')
    return len(rows)

print('Helper functions ready.')


Helper functions ready.


In [0]:
print('Generating drivers master file...')

vehicle_types = ['Auto','Sedan','SUV']
v_weights     = [0.40, 0.45, 0.15]

drivers = []
for i in range(TOTAL_DRIVERS):
    city = random.choices(CITIES, weights=CITY_WEIGHTS)[0]
    vtype = random.choices(vehicle_types, weights=v_weights)[0]
    status = 'active' if random.random() < 0.95 else 'inactive'
    join_days_ago = random.randint(365, 3*365)
    join_date = (START_DATE - timedelta(days=join_days_ago)).isoformat()
    drivers.append({
        'driver_id':        str(uuid.uuid4()),
        'name':             fake.name(),
        'city':             city,          # clean names for drivers
        'vehicle_type':     vtype,
        'rating':           round(random.uniform(3.5, 5.0), 2),
        'trips_completed':  random.randint(50, 8000),
        'joining_date':     join_date,
        'status':           status,
    })

driver_df = pd.DataFrame(drivers)

# Write directly to Volume path — no /tmp/ buffer, no dbutils.fs.cp()
driver_df.to_csv(f'{BASE_PATH}/drivers/drivers_master.csv', index=False)

# Keep pools in memory for relationship enforcement
active_drivers = driver_df[driver_df['status']=='active'][['driver_id','city','vehicle_type']].to_dict('records')
driver_by_city = {c:[d for d in active_drivers if d['city']==c] for c in CITIES}

print(f'  drivers_master.csv written — {len(drivers):,} drivers ({len(active_drivers):,} active)')
print(f'  Vehicle mix: Auto={sum(1 for d in drivers if d["vehicle_type"]=="Auto")} | Sedan={sum(1 for d in drivers if d["vehicle_type"]=="Sedan")} | SUV={sum(1 for d in drivers if d["vehicle_type"]=="SUV")}')


Generating drivers master file...
  drivers_master.csv written — 20,000 drivers (18,996 active)
  Vehicle mix: Auto=7953 | Sedan=9020 | SUV=3027


In [0]:
print('Generating customer files...')

all_customer_ids   = []    # grows daily — used for trip/event sampling
churned_map        = {}    # customer_id -> churn_date (if churned)
customer_ab_group  = {}    # customer_id -> 'control' or 'variant'
customer_city_map  = {}    # customer_id -> city
customer_device_map = {}   # customer_id -> device_type
referral_pool      = []    # customers who came via referral — for referred_by

total_customers_written = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')

    # Scale new customers — weekends fewer signups
    n_new = int(NEW_CUSTOMERS_PER_DAY * day_demand_multiplier(current_date))
    n_new = max(200, n_new + random.randint(-50, 50))

    rows = []
    for _ in range(n_new):
        city    = random.choices(CITIES, weights=CITY_WEIGHTS)[0]
        device  = random.choices(['Android','iOS','Web'], weights=[0.70,0.25,0.05])[0]
        ref_src = random.choices(['Ads','Organic','Referral'], weights=[0.40,0.35,0.25])[0]
        cid     = str(uuid.uuid4())

        # Churn logic — city-specific rates
        churn_rate = CITY_CHURN[city]
        is_churned = random.random() < churn_rate
        # Churn happens in months 1-4
        churn_dt = None
        if is_churned:
            days_till_churn = random.randint(14, 120)
            churn_dt = (current_date + timedelta(days=days_till_churn))
            if churn_dt > END_DATE: churn_dt = None; is_churned = False

        # Referred by — pick from existing referral pool
        referred_by = None
        if ref_src == 'Referral' and referral_pool:
            referred_by = random.choice(referral_pool)

        # Rating — ~0.5% get invalid rating > 5
        rating = round(random.uniform(3.5, 5.0), 1)
        if random.random() < 0.005: rating = round(random.uniform(5.1, 7.0), 1)
        # ~0.5% get NULL rating
        if random.random() < 0.005: rating = None

        row = {
            'customer_id':             cid,
            'name':                    fake.name(),
            'city':                    dirty_city(city),   # dirty city names here only
            'signup_date':             current_date.isoformat(),
            'device_type':             device,
            'referral_source':         ref_src,
            'referred_by_customer_id': referred_by,
            'rating':                  rating,
            'is_churned':              is_churned,
            'churn_date':              churn_dt.isoformat() if churn_dt else None,
        }

        # ~1% duplicate rows
        rows.append(row)
        if random.random() < 0.01: rows.append(row.copy())

        # Store in memory pools
        all_customer_ids.append(cid)
        customer_city_map[cid]   = city
        customer_device_map[cid] = device
        customer_ab_group[cid]   = 'variant' if hash(cid) % 2 == 0 else 'control'
        if is_churned and churn_dt: churned_map[cid] = churn_dt
        if ref_src in ['Ads','Organic'] and len(referral_pool) < 50000:
            referral_pool.append(cid)

    fpath = f'{BASE_PATH}/customers/customers_{ds}.csv'
    pd.DataFrame(rows).to_csv(fpath, index=False)
    n_written = len(rows)
    total_customers_written += n_written

    if day_offset % 15 == 0 or day_offset == TOTAL_DAYS-1:
        print(f'  Day {day_offset+1:>3}/{TOTAL_DAYS} — {current_date} — {n_written} customers  (total pool: {len(all_customer_ids):,})')

print(f'\nCustomers done. Total written: {total_customers_written:,}  |  Unique pool: {len(all_customer_ids):,}  |  Churned: {len(churned_map):,}')


Generating customer files...
  Day   1/182 — 2025-09-01 — 518 customers  (total pool: 516)
  Day  16/182 — 2025-09-16 — 532 customers  (total pool: 8,085)
  Day  31/182 — 2025-10-01 — 548 customers  (total pool: 15,224)
  Day  46/182 — 2025-10-16 — 461 customers  (total pool: 22,333)
  Day  61/182 — 2025-10-31 — 528 customers  (total pool: 30,069)
  Day  76/182 — 2025-11-15 — 376 customers  (total pool: 37,217)
  Day  91/182 — 2025-11-30 — 381 customers  (total pool: 44,335)
  Day 106/182 — 2025-12-15 — 509 customers  (total pool: 51,553)
  Day 121/182 — 2025-12-30 — 551 customers  (total pool: 58,966)
  Day 136/182 — 2026-01-14 — 703 customers  (total pool: 66,808)
  Day 151/182 — 2026-01-29 — 538 customers  (total pool: 74,051)
  Day 166/182 — 2026-02-13 — 617 customers  (total pool: 81,249)
  Day 181/182 — 2026-02-28 — 363 customers  (total pool: 88,406)
  Day 182/182 — 2026-03-01 — 638 customers  (total pool: 89,041)

Customers done. Total written: 89,963  |  Unique pool: 89,041  |

In [0]:
print('Generating trip files...')

trip_ids_by_date  = {}   # date -> list of completed trip_ids (for payments)
trip_fare_by_id   = {}   # trip_id -> fare_amount (for payment matching)
trip_cust_by_id   = {}   # trip_id -> customer_id (for payment matching)
total_trips       = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')

    multiplier = day_demand_multiplier(current_date)
    n_trips    = int(TRIPS_PER_DAY_BASE * multiplier) + random.randint(-2000, 2000)

    # Only sample from customers who signed up before today
    available_customers = [
        cid for cid in all_customer_ids
        if cid not in churned_map or churned_map[cid] > current_date
    ]
    if len(available_customers) < 1000:
        available_customers = all_customer_ids[:1000]  # fallback for day 1

    rows = []
    completed_trips = []

    for _ in range(n_trips):
        city = random.choices(CITIES, weights=CITY_WEIGHTS)[0]

        # Pick customer from available pool for this city (or any city)
        cid = random.choice(available_customers)
        actual_city = customer_city_map.get(cid, city)
        # 70% trips in home city, 30% in random city
        trip_city = actual_city if random.random() < 0.70 else random.choice(CITIES)

        # Driver — prefer same city, fall back to any
        city_drivers = driver_by_city.get(trip_city, active_drivers)
        if not city_drivers: city_drivers = active_drivers
        driver = random.choice(city_drivers)

        # Time — weighted toward commute hours on weekdays
        if is_weekend(current_date) or is_holiday(current_date):
            hour = random.choices(range(24), weights=[
                1,1,1,1,1,1,2,3,5,6,6,7,7,7,6,5,5,6,7,6,5,4,3,2
            ])[0]
        else:
            hour = random.choices(range(24), weights=[
                1,1,1,1,1,1,3,7,9,6,5,5,5,5,5,5,6,9,9,7,5,4,3,2
            ])[0]
        minute = random.randint(0,59)
        pickup_dt = datetime(current_date.year, current_date.month, current_date.day, hour, minute)

        duration_min  = random.randint(10, 40)
        drop_dt       = pickup_dt + timedelta(minutes=duration_min)
        weather       = get_weather(trip_city, current_date.month)
        surge         = get_surge(hour, weather)

        # Distance — ~0.3% negative (data quality issue)
        distance = round(max(0.5, np.random.normal(7.0, 3.0)), 2)
        if random.random() < 0.003: distance = -round(random.uniform(0.1, 5.0), 2)

        # Ghost trip (fraud) — ~0.2%
        is_ghost = random.random() < 0.002
        if is_ghost: distance = 0.0

        fare = calc_fare(abs(distance), surge, driver['vehicle_type'])

        # Fare inflation fraud — ~0.1%
        is_fare_inflated = False
        if random.random() < 0.001:
            fare = fare * random.uniform(2.5, 3.5)
            is_fare_inflated = True

        status = random.choices(
            ['completed','cancelled'],
            weights=[0.88 if trip_city=='Pune' else 0.92,
                     0.12 if trip_city=='Pune' else 0.08]
        )[0]

        trip_id = str(uuid.uuid4())
        row = {
            'trip_id':          trip_id,
            'customer_id':      cid,
            'driver_id':        driver['driver_id'],
            'city':             trip_city,          # clean city name in trips
            'pickup_time':      pickup_dt.strftime('%Y-%m-%d %H:%M:%S'),
            'drop_time':        drop_dt.strftime('%Y-%m-%d %H:%M:%S'),
            'distance_km':      distance,
            'fare_amount':      round(fare, 2),
            'surge_multiplier': surge,
            'trip_status':      status,
            'vehicle_type':     driver['vehicle_type'],
            'weather':          weather,
            'is_ghost_trip':    is_ghost,
            'is_fare_inflated': is_fare_inflated,
            'day_type':         'holiday' if is_holiday(current_date) else ('weekend' if is_weekend(current_date) else 'weekday'),
        }
        rows.append(row)
        if status == 'completed':
            completed_trips.append(trip_id)
            trip_fare_by_id[trip_id] = fare
            trip_cust_by_id[trip_id] = cid

    trip_ids_by_date[current_date] = completed_trips

    fpath = f'{BASE_PATH}/trips/trips_{ds}.json'
    with open(fpath, 'w') as f:
        for r in rows: f.write(json.dumps(r) + '\n')
    n_written = len(rows)

    if day_offset % 15 == 0 or day_offset == TOTAL_DAYS-1:
        wtype = 'HOLIDAY' if is_holiday(current_date) else ('WEEKEND' if is_weekend(current_date) else 'weekday')
        print(f'  Day {day_offset+1:>3}/{TOTAL_DAYS} — {current_date} [{wtype:7}] x{multiplier:.2f} — {n_written:,} trips ({len(completed_trips):,} completed)')

print(f'\nTrips done. Total: {total_trips:,}')


Generating trip files...
  Day   1/182 — 2025-09-01 [weekday] x1.00 — 119,826 trips (109,471 completed)
  Day  16/182 — 2025-09-16 [weekday] x1.00 — 118,999 trips (109,049 completed)
  Day  31/182 — 2025-10-01 [weekday] x1.00 — 118,205 trips (108,060 completed)
  Day  46/182 — 2025-10-16 [weekday] x1.00 — 118,628 trips (108,435 completed)
  Day  61/182 — 2025-10-31 [weekday] x1.15 — 136,116 trips (124,538 completed)
  Day  76/182 — 2025-11-15 [WEEKEND] x0.75 — 91,549 trips (83,815 completed)
  Day  91/182 — 2025-11-30 [WEEKEND] x0.75 — 91,459 trips (83,705 completed)
  Day 106/182 — 2025-12-15 [weekday] x1.00 — 118,492 trips (108,302 completed)
  Day 121/182 — 2025-12-30 [weekday] x1.00 — 121,342 trips (111,091 completed)
  Day 136/182 — 2026-01-14 [HOLIDAY] x1.30 — 155,197 trips (142,379 completed)
  Day 151/182 — 2026-01-29 [weekday] x1.00 — 119,003 trips (108,744 completed)
  Day 166/182 — 2026-02-13 [weekday] x1.15 — 138,764 trips (126,987 completed)
  Day 181/182 — 2026-02-28 [WEE

In [0]:
print('Generating payment files...')

# Build fraud card pool — same card_hash across 3-5 accounts
n_fraud_cards = int(len(all_customer_ids) * 0.002)  # 0.2% of customers involved in card fraud
fraud_cards   = [str(uuid.uuid4())[:16] for _ in range(n_fraud_cards // 3)]
fraud_card_customers = {}
for fc in fraud_cards:
    n_accounts = random.randint(3, 5)
    fraud_card_customers[fc] = random.sample(all_customer_ids, min(n_accounts, len(all_customer_ids)))

# customer -> card_hash mapping (for fraud)
cust_to_fraud_card = {}
for card, custs in fraud_card_customers.items():
    for c in custs: cust_to_fraud_card[c] = card

total_payments = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')
    completed = trip_ids_by_date.get(current_date, [])

    rows = []
    for tid in completed:
        fare  = trip_fare_by_id.get(tid, 150.0)
        cid   = trip_cust_by_id.get(tid, random.choice(all_customer_ids))
        method= random.choices(['UPI','Credit Card','Wallet','Cash'], weights=[0.55,0.20,0.15,0.10])[0]

        # card_hash — only for Credit Card
        card_hash = None
        if method == 'Credit Card':
            if cid in cust_to_fraud_card:
                card_hash = cust_to_fraud_card[cid]  # fraud signal
            else:
                card_hash = str(uuid.uuid4())[:16]

        # ~0.2% fare mismatch
        amount = fare
        if random.random() < 0.002:
            amount = round(fare * random.uniform(0.7, 1.4), 2)

        # Payment time — within 5 mins of drop
        pay_offset = random.randint(0, 300)
        pay_status = random.choices(['success','failed'], weights=[0.97,0.03])[0]

        rows.append({
            'payment_id':       str(uuid.uuid4()),
            'trip_id':          tid,
            'customer_id':      cid,
            'amount':           round(amount, 2),
            'payment_method':   method,
            'payment_timestamp':f'{current_date.isoformat()} 00:{pay_offset//60:02}:{pay_offset%60:02}',
            'payment_status':   pay_status,
            'card_hash':        card_hash,
            'is_fraud_card':    cid in cust_to_fraud_card,
        })

    # ~0.3% orphan payments (no matching trip)
    n_orphan = max(1, int(len(rows) * 0.003))
    for _ in range(n_orphan):
        rows.append({
            'payment_id':       str(uuid.uuid4()),
            'trip_id':          str(uuid.uuid4()),   # non-existent trip
            'customer_id':      random.choice(all_customer_ids),
            'amount':           round(random.uniform(50,300), 2),
            'payment_method':   'UPI',
            'payment_timestamp':current_date.isoformat(),
            'payment_status':   'success',
            'card_hash':        None,
            'is_fraud_card':    False,
        })

    fpath = f'{BASE_PATH}/payments/payments_{ds}.csv'
    pd.DataFrame(rows).to_csv(fpath, index=False)
    n_written = len(rows)

    if day_offset % 15 == 0 or day_offset == TOTAL_DAYS-1:
        print(f'  Day {day_offset+1:>3}/{TOTAL_DAYS} — {current_date} — {n_written:,} payments')

print(f'\nPayments done. Total: {total_payments:,}')


Generating payment files...
  Day   1/182 — 2025-09-01 — 109,799 payments
  Day  16/182 — 2025-09-16 — 109,376 payments
  Day  31/182 — 2025-10-01 — 108,384 payments
  Day  46/182 — 2025-10-16 — 108,760 payments
  Day  61/182 — 2025-10-31 — 124,911 payments
  Day  76/182 — 2025-11-15 — 84,066 payments
  Day  91/182 — 2025-11-30 — 83,956 payments
  Day 106/182 — 2025-12-15 — 108,626 payments
  Day 121/182 — 2025-12-30 — 111,424 payments
  Day 136/182 — 2026-01-14 — 142,806 payments
  Day 151/182 — 2026-01-29 — 109,070 payments
  Day 166/182 — 2026-02-13 — 127,367 payments
  Day 181/182 — 2026-02-28 — 82,116 payments
  Day 182/182 — 2026-03-01 — 142,431 payments

Payments done. Total: 0


In [0]:
print('Generating clickstream event files...')

total_events = 0

for day_offset in range(TOTAL_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    ds = current_date.strftime('%Y_%m_%d')

    multiplier = day_demand_multiplier(current_date)
    n_events   = int(EVENTS_PER_DAY_BASE * multiplier) + random.randint(-5000, 5000)

    # Only active customers generate events
    active_customers = [
        cid for cid in all_customer_ids
        if cid not in churned_map or churned_map[cid] > current_date
    ]
    if len(active_customers) < 500:
        active_customers = all_customer_ids[:500]

    rows = []
    n_sessions = n_events // 4  # avg ~4 events per session

    for _ in range(n_sessions):
        cid        = random.choice(active_customers)
        city       = customer_city_map.get(cid, random.choice(CITIES))
        device     = customer_device_map.get(cid, 'Android')
        ab_group   = customer_ab_group.get(cid, 'control')
        session_id = str(uuid.uuid4())

        hour = random.randint(7, 23)
        base_ts = datetime(current_date.year, current_date.month, current_date.day, hour, random.randint(0,59))

        # Session event sequence — always starts with app_open
        # Variant group has 8% higher confirm_booking rate
        confirm_weight = 0.12 if ab_group == 'variant' else 0.10
        cancel_weight  = 0.04 if ab_group == 'variant' else 0.05

        session_flow = ['app_open']
        if random.random() < 0.65: session_flow.append('search_ride')
        if random.random() < 0.60 and len(session_flow)>1: session_flow.append('view_price')
        r = random.random()
        if r < confirm_weight and len(session_flow)>2: session_flow.append('confirm_booking')
        elif r < confirm_weight + cancel_weight: session_flow.append('cancel_booking')

        for seq, etype in enumerate(session_flow):
            ts = base_ts + timedelta(seconds=seq*random.randint(5,60))
            rows.append({
                'event_id':       str(uuid.uuid4()),
                'customer_id':    cid,
                'session_id':     session_id,
                'event_type':     etype,
                'event_timestamp':ts.strftime('%Y-%m-%d %H:%M:%S'),
                'city':           city,
                'device_type':    device,
                'ab_test_group':  ab_group,
                'ab_test_name':   'pricing_ui_v2',
            })

    # fpath = f'{BASE_PATH}/clickstream/events_{ds}.json'
    # n_written = write_jsonl_to_dbfs(rows, fpath)
    # total_events += n_written
    fpath = f'{BASE_PATH}/clickstream/events_{ds}.json'
    with open(fpath, 'w') as f:
        for r in rows: f.write(json.dumps(r) + '\n')
    n_written = len(rows)

    if day_offset % 15 == 0 or day_offset == TOTAL_DAYS-1:
        wtype = 'HOLIDAY' if is_holiday(current_date) else ('WEEKEND' if is_weekend(current_date) else 'weekday')
        print(f'  Day {day_offset+1:>3}/{TOTAL_DAYS} — {current_date} [{wtype:7}] — {n_written:,} events')

print(f'\nClickstream done. Total: {total_events:,}')


Generating clickstream event files...
  Day   1/182 — 2025-09-01 [weekday] — 136,495 events
  Day  16/182 — 2025-09-16 [weekday] — 139,947 events
  Day  31/182 — 2025-10-01 [weekday] — 136,831 events
  Day  46/182 — 2025-10-16 [weekday] — 136,286 events
  Day  61/182 — 2025-10-31 [weekday] — 156,110 events
  Day  76/182 — 2025-11-15 [WEEKEND] — 104,989 events
  Day  91/182 — 2025-11-30 [WEEKEND] — 101,633 events
  Day 106/182 — 2025-12-15 [weekday] — 135,975 events
  Day 121/182 — 2025-12-30 [weekday] — 136,463 events
  Day 136/182 — 2026-01-14 [HOLIDAY] — 175,462 events
  Day 151/182 — 2026-01-29 [weekday] — 138,881 events
  Day 166/182 — 2026-02-13 [weekday] — 156,325 events
  Day 181/182 — 2026-02-28 [WEEKEND] — 103,424 events
  Day 182/182 — 2026-03-01 [HOLIDAY] — 177,197 events

Clickstream done. Total: 0


In [0]:
dbutils.fs.ls("dbfs:/")

[FileInfo(path='dbfs:/Volumes/', name='Volumes/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/Workspace/', name='Workspace/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-datasets/', name='databricks-datasets/', size=0, modificationTime=0)]

In [0]:
# # Read raw files back into Spark and register as Delta tables in urbanride.bronze
# # This makes files queryable via SQL and ready for Silver transformation notebooks
# print('Registering Bronze Delta tables in Unity Catalog...')

# from pyspark.sql.types import *

# # ── DRIVERS ──────────────────────────────────────────────────────────────
# print('  Registering bronze.drivers...')
# df_drivers = spark.read.option('header','true').option('inferSchema','true') \
#     .csv(f'{BASE_PATH}/drivers/')
# df_drivers.writeTo(f'{BRONZE}.drivers').using('delta').createOrReplace()
# print(f'    Rows: {df_drivers.count():,}')

# # ── CUSTOMERS ─────────────────────────────────────────────────────────────
# print('  Registering bronze.customers...')
# df_customers = spark.read.option('header','true').option('inferSchema','true') \
#     .csv(f'{BASE_PATH}/customers/')
# df_customers.writeTo(f'{BRONZE}.customers').using('delta').createOrReplace()
# print(f'    Rows: {df_customers.count():,}')

# # ── TRIPS ─────────────────────────────────────────────────────────────────
# print('  Registering bronze.trips...')
# df_trips = spark.read.option('multiline','false').json(f'{BASE_PATH}/trips/')
# df_trips.writeTo(f'{BRONZE}.trips').using('delta').createOrReplace()
# print(f'    Rows: {df_trips.count():,}')

# # ── PAYMENTS ──────────────────────────────────────────────────────────────
# print('  Registering bronze.payments...')
# df_payments = spark.read.option('header','true').option('inferSchema','true') \
#     .csv(f'{BASE_PATH}/payments/')
# df_payments.writeTo(f'{BRONZE}.payments').using('delta').createOrReplace()
# print(f'    Rows: {df_payments.count():,}')

# # ── CLICKSTREAM ───────────────────────────────────────────────────────────
# print('  Registering bronze.clickstream...')
# df_events = spark.read.option('multiline','false').json(f'{BASE_PATH}/clickstream/')
# df_events.writeTo(f'{BRONZE}.clickstream').using('delta').createOrReplace()
# print(f'    Rows: {df_events.count():,}')

# print()
# print('All Bronze tables registered in Unity Catalog.')


In [0]:
# print('=== URBANRIDE DATA GENERATION — FINAL SUMMARY ===')
# print()

# tables = [
#     (f'{BRONZE}.drivers',     'Drivers master'),
#     (f'{BRONZE}.customers',   'Customers (all days)'),
#     (f'{BRONZE}.trips',       'Trips (all days)'),
#     (f'{BRONZE}.payments',    'Payments (all days)'),
#     (f'{BRONZE}.clickstream', 'Clickstream (all days)'),
# ]

# print(f'  {"Table":<35} {"Rows":>15}  Description')
# print('  ' + '-'*70)
# for tbl, desc in tables:
#     try:
#         n = spark.table(tbl).count()
#         print(f'  {tbl:<35} {n:>15,}  {desc}')
#     except Exception as e:
#         print(f'  {tbl:<35} {"ERROR":>15}  {e}')

# print()
# print(f'  Date range  : {START_DATE}  to  {END_DATE}  ({TOTAL_DAYS} days)')
# print(f'  Catalog     : {CATALOG}')
# print(f'  Schemas     : bronze · silver · gold')
# print()
# print('Data quality issues embedded:')
# issues = [
#     '  ~1%    duplicate customer rows',
#     '  ~0.3%  negative trip distances',
#     '  ~20%   dirty city names in customers',
#     '  ~0.5%  invalid customer ratings (> 5.0)',
#     '  ~0.3%  orphan payments (no matching trip)',
#     '  ~0.5%  shared card_hash across 3+ accounts (fraud)',
#     '  ~0.2%  ghost trips (distance = 0, status = completed)',
#     '  ~0.1%  fare-inflated trips',
# ]
# for i in issues: print(i)
# print()
# print('ML signals embedded:')
# signals = [
#     '  Churn     — is_churned + churn_date + declining activity before churn',
#     '  Fraud     — is_fraud_card + is_ghost_trip + is_fare_inflated + orphan payments',
#     '  Demand    — weather + day_type + hour in trip data for forecasting',
#     '  A/B test  — ab_test_group (control/variant) with 8% variant lift in bookings',
# ]
# for s in signals: print(s)
# print()
# print('Next step: Build Silver transformation notebooks')
# print('  Customer domain  -> silver.customers_clean')
# print('  Ride domain      -> silver.trips_clean')
# print('  Finance domain   -> silver.payments_clean')
# print('  Product domain   -> silver.clickstream_clean')
